## Contornos

In [ ]:
#Si queremos que las imágenes sean mostradas en una ventana emergente quitar el inline

# OpenCV-Python utiliza NumPy para el manejo de imágenes
import numpy as np
# cv2 es el módulo python para acceder a OpenCV 
import cv2 as cv
# Usamos las poderosas herramientas de graficación de matplotlib para mostrar imágenes, perfiles, histogramas, etc
import supervision as sv

In [ ]:
# Leemos la imagen y la binarizamos
#==================================
img = cv.imread('engranaje.jpg')

imgray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

# Binarizamos con Otzu
ret, img_bin = cv.threshold(imgray,30,255,cv.THRESH_BINARY+cv.THRESH_OTSU)

sv.plot_images_grid(images=[img, img_bin], titles=['Imagen original', 'Imagen binarizada'], grid_size=(1, 2))
cv.imshow('Original',img_bin)
#cv.imshow('Original_gris',img)

### Cálculo

El cálculo de contornos tiene un parámetro obligatorio de modo:

- cv.RETR_LIST: Contornos sin relación de jerarquías
- cv.RETR_CCOMP: Contornos con dos niveles de jerarquía --> Contorno externos y contornos internos
- cv.RETR_TREE: Mapeo completo de jerarquías, inclusive para contornos anidados.

Y un parámetro optativo de método:

- cv.CHAIN_APPROX_NONE: Devuelve todos los puntos del contorno
- cv.CHAIN_APPROX_SIMPLE: Comprime los segmentos horizontales y verticales (devuelve solo los extremos)
- cv.CHAIN_APPROX_TC89_L1: Aproximación según el algoritmo de Ten-Chin
- cv.CHAIN_APPROX_TC89_KCOS: Aproximación según el algoritmo de Ten-Chin


In [ ]:
# Calculamos los contornos
contours, hierarchy = cv.findContours(img_bin, cv.RETR_TREE, cv.CHAIN_APPROX_NONE)
# Para el caso del engranaje no hay nada que pueda simplificar
#im2, contours, hierarchy = cv.findContours(img_bin, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)

# Me fijo cuántos contornos se encontraron
print(len(contours))

### Dibujo

Podemos dibujar todos los contornos encontrados o contornos individuales

In [ ]:
# Dibujamos todos los contornos (indicado con el -1)
#---------------------------------------------------
img_out = img.copy()
cv.drawContours(img_out, contours, -1, (0, 255, 0), 1)
sv.plot_image(img_out)


In [ ]:


# Dibujamos un contorno en particular
#------------------------------------
img_out = img.copy()
cv.drawContours(img_out, contours, 1, (0,255,0), 3)
# Otra manera de elegir un contorno en particular
cnt = contours[4]
cv.drawContours(img_out, cnt, -1, (0,255,255), 3)

sv.plot_image(img_out)

### Jerarquías

Es posible pedir mayor o menor información de jerarquías según sea necesario. Para eso disponemos del parámetro de modo

- cv.RETR_LIST: Contornos sin relación de jerarquías
- cv.RETR_CCOMP: Contornos con dos niveles de jerarquía --> Contorno externos y contornos internos
- cv.RETR_TREE: Mapeo completo de jerarquías, inclusive para contornos anidados.


- __Orden:__ [Next, Previous, First_Child, Parent]

In [ ]:
# Contornos sin relación de jerarquía
contours, hierarchy = cv.findContours(img_bin, cv.RETR_LIST, cv.CHAIN_APPROX_NONE)
print('Lista: {} \n'.format(hierarchy))

# Contornos en dos niveles de jerarquía
contours, hierarchy = cv.findContours(img_bin, cv.RETR_CCOMP, cv.CHAIN_APPROX_NONE)
print('Dos niveles: {} \n'.format(hierarchy))

# Contornos con mapeo completo de jerarquía
contours, hierarchy = cv.findContours(img_bin, cv.RETR_TREE, cv.CHAIN_APPROX_NONE)
print('Full: {} \n'.format(hierarchy))

### Operaciones

Veamos algunas operaciones que podemos hacer con los contornos

#### Perímetro y Área

In [ ]:
# Veamos qué longitud tienen por ejemplo los contornos hallados

for cnt in contours:
    print('Longitud: {} - Área: {}'.format(cv.arcLength(cnt, False), cv.contourArea(cnt)))
    

In [ ]:
#Veámoslos...
img_out = img.copy()

# De arriba ya vemos que (sin mirar las jerarquías) tenemos tres perímetros áreas mayores: 0, 1, y 13

CNTR_IDX = [0, 1, 13]
for idx in CNTR_IDX:
    cnt = contours[idx]
    cv.drawContours(img_out, [cnt], 0, (0,255,0), 3)

cv.drawContours(img_out, [cnt], 0, (0,255,0), 3)
sv.plot_image(img_out)

#### Centroide

Para encontrar el centroide debemos calcular primero los momentos

In [ ]:
# Calculamos primero los momentos
cnt = contours[2]
M = cv.moments(cnt)
#print( M )

# Ahora calculamos el centroide
cx = int(M['m10']/M['m00'])
cy = int(M['m01']/M['m00'])

print('Centroide: [{},{}]'.format(cx,cy))

cv.circle(img_out, (cx, cy), 7, (255, 0, 0), -1)
sv.plot_image(img_out)

# Debería parecerse bastante al centro de la imagen...
size = img.shape
print('Tamaño imagen: {}'.format(size))
print('Centro imagen: [{},{}]'.format(int(size[0]/2), int(size[1]/2)))

#### Convex Hull

Es una aproximación de casquete convexo sobre la geometría indicada

In [ ]:
# Lo calculamos...
cnt = contours[1]
hull = cv.convexHull(cnt)

# Lo dibujamos
img_out = img.copy()
cv.drawContours(img_out, [hull], -1, (0, 0, 255), 2)

sv.plot_image(img_out)


#### Bounding Box

In [ ]:
# Calculamos...
cnt = contours[1]
x,y,w,h = cv.boundingRect(cnt)

# Lo dibujamos...
img_out = img.copy()
cv.rectangle(img_out,(x,y),(x+w,y+h),(255,0,0),3)
sv.plot_image(img_out)

#### Bounding Box Rotado

A veces el rectángulo que mejor ajusta está rotado.

In [ ]:
# Lo calculamos...
cnt = contours[13]
rect = cv.minAreaRect(cnt)
box = cv.boxPoints(rect)

# Lo dibujamos...
img_out = img.copy()
cv.drawContours(img_out,[box.astype(int)],0,(255,0,0),2)
sv.plot_image(img_out)

#### Ajuste de un círculo

In [ ]:
# Lo calculamos...
cnt = contours[13]
(x,y),radius = cv.minEnclosingCircle(cnt)
center = (int(x),int(y))
radius = int(radius)

# Lo dibujamos...
img_out = img.copy()
cv.circle(img_out,center,radius,(0,0,255),2)
sv.plot_image(img_out)

#### Ajuste de una elipse

In [ ]:
# Lo calculamos...
cnt = contours[13]
ellipse = cv.fitEllipse(cnt)

# Lo dibujamos...
img_out = img.copy()
cv.ellipse(img_out,ellipse,(0,255,0),2)
sv.plot_image(img_out)

#### Ajuste de recta

In [ ]:
rows,cols = img.shape[:2]
# Lo calculamos...
cnt = contours[1]
[vx,vy,x,y] = cv.fitLine(cnt, cv.DIST_L2,0,0.01,0.01)
lefty = int((-x*vy/vx) + y)
righty = int(((cols-x)*vy/vx)+y)

# Lo dibujamos...
img_out = img.copy()
cv.drawContours(img_out,[cnt],0,(255,255,0),2)
cv.line(img_out,(cols-1,righty),(0,lefty),(0,255,0),2)
sv.plot_image(img_out)

### Propiedades de los contornos


Además del centroide, área y perímetro tenemos otras magnitudes a evaluar con los contornos

- Relación de aspecto
- Extensión
- Solidez
- Diámetro equivalente
- Orientación

In [ ]:
# Elegimos el contorno a evaluar
cnt = contours[1]

# Relación de aspecto
x,y,w,h = cv.boundingRect(cnt)
aspect_ratio = float(w)/h
print('Relación de aspecto: {}'.format(aspect_ratio))

# Extensión (Relación entre el área del objeto y del bounding box)
area = cv.contourArea(cnt)
x,y,w,h = cv.boundingRect(cnt)
rect_area = w*h
extent = float(area)/rect_area
print('Extensión:           {}'.format(extent))

# Solidez (Relación entre el área del objeto y del casquete convexo)
area = cv.contourArea(cnt)
hull = cv.convexHull(cnt)
hull_area = cv.contourArea(hull)
solidity = float(area)/hull_area
print('Solidez:             {}'.format(solidity))

# Diámetro equivalente (Diámetro de un círculo que tendría la misma área que el objeto)
area = cv.contourArea(cnt)
equi_diameter = np.sqrt(4*area/np.pi)
print('Diámetro equivalente:{}'.format(equi_diameter))

# Orientación (Con respecto a la horizontal)
(x,y),(MA,ma),angle = cv.fitEllipse(cnt)
print('Orientación:         {}'.format(angle))

#### Máscaras

Definido un contorno muchas veces es útil generar una máscara alrededor del mismo para análisis localizados

In [ ]:
# Elegimos el contorno a evaluar
cnt = contours[1]

# Armamos la base de la máscara en función de la imagen original
mask = np.zeros(imgray.shape, np.uint8)

# Luego marcamos únicamente el contorno (el último parámetro en -1 indica "llenar" el contorno)
cv.drawContours(mask,[cnt],0,255,-1)
pixelpoints = np.transpose(np.nonzero(mask))

sv.plot_image(mask)

In [ ]:
# A partir de esta máscara podemos calcular valores sobre la imagen original

# Intensidades mínimas/máximas
min_val, max_val, min_loc, max_loc = cv.minMaxLoc(imgray,mask = mask)
print('[min val, max val, min loc, max loc]: [{},{},{},{}]'.format(min_val, max_val, min_loc, max_loc))


# Valor de intensidad media
mean_val = cv.mean(imgray, mask = mask)
print('Promedio: {}'.format(mean_val[0]))

# Etc.

#### Defectos de convexidad

Analizar la diferencia entre un objeto y el casquete convexo que lo contiene

In [ ]:
# Elegimos el contorno a evaluar
cnt = contours[1]

# Calculamos el casquete y diferencias
hull = cv.convexHull(cnt,returnPoints = False)
defects = cv.convexityDefects(cnt, hull)

# Dibujamos...
img_out = img.copy()

for i in range(defects.shape[0]):
    s,e,f,d = defects[i,0]
    start = tuple(cnt[s][0])
    end = tuple(cnt[e][0])
    far = tuple(cnt[f][0])
    cv.line(img_out,start,end,[0,255,0],2)
    cv.circle(img_out,far,1,[0,0,255],-1)

sv.plot_image(img_out)

### Coincidencia de figuras

#### Verificación contra contorno conocido

In [ ]:
# Contorno de referencia
# 1. obtener un contorno de referencia (por ejemplo el contorno 1)
# 2. comparar el resto de los contornos con el contorno de referencia 
# utilizando la función cv.matchShapes()

refcnt = contours[1]

for cnt in contours:
    ret = cv.matchShapes(refcnt, cnt,1,0.0)
    print(f'Coincidencia: {ret}')